# Parte 1 — Análise Exploratória dos Chamados (Q1)

## Contexto

A **Central 1746** recebe chamados de cidadãos que precisam ser encaminhados para o serviço correto. O **modelo A** (em produção) e o **modelo B** (candidato) classificam automaticamente o texto do chamado em categorias de serviço.

Este notebook explora o corpus de **5.000 chamados rotulados** (`dados/chamados_com_predicoes.csv`) para entender características dos dados que **impactam a avaliação dos classificadores**.

> **Nota:** os dados são sintéticos — simulam o comportamento estatístico de um sistema real, sem uso de dados de cidadãos.

## Objetivo

Identificar padrões no corpus (categorias, textos, canal, etc.) relevantes para o **problema de classificação**, preparando a auditoria do modelo A (Parte 2) e a comparação A vs B (Parte 3).

## Dados utilizados

| Coluna | Descrição |
|--------|-----------|
| `texto` | Texto do chamado (input dos modelos) |
| `categoria_real` | Rótulo humano (ground truth) |
| `canal`, `bairro`, `data_abertura` | Metadados do chamado |
| `pred_modelo_a/b`, `conf_modelo_a/b` | Predições dos modelos *(analisadas nas Partes 2 e 3)* |

## Roteiro da análise

1. **Panorama geral** — dimensões, qualidade e período dos dados  
2. **Distribuição de categorias** — desbalanceamento entre classes  
3. **Características dos textos** — comprimento e variabilidade do input  
4. **Padrões por canal** — diferenças na forma de entrada dos chamados  
5. **Síntese** — 3–5 achados principais e relevância para classificação

## Setup

In [45]:
import pandas as pd
from pathlib import Path
from IPython.display import display

# Caminho para o arquivo CSV
ROOT = Path("..").resolve()
DATA_PATH = ROOT / "dados" / "chamados_com_predicoes.csv"

## Carregamento e validação inicial

Carregamos o CSV, validamos dimensões e ausência de nulos, e criamos
colunas derivadas para as análises seguintes.

In [46]:
# Carregando o arquivo CSV
df = pd.read_csv(DATA_PATH)

# Colunas derivadas para as análises seguintes
df["texto_len"] = df["texto"].str.len()
df["data_abertura"] = pd.to_datetime(df["data_abertura"])

# Sanity checks (estrutura e domínio)
assert df.shape == (5000, 11)
assert df.isna().sum().sum() == 0
assert (df["id_chamado"]).is_unique
assert (df["texto_len"] > 0).all()
assert (df["conf_modelo_a"]).between(0, 1).all()
assert (df["conf_modelo_b"]).between(0, 1).all()

# Exibindo estrutura e as primeiras linhas do DataFrame
df.info()
display(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id_chamado      5000 non-null   int64         
 1   data_abertura   5000 non-null   datetime64[us]
 2   bairro          5000 non-null   str           
 3   canal           5000 non-null   str           
 4   texto           5000 non-null   str           
 5   categoria_real  5000 non-null   str           
 6   pred_modelo_a   5000 non-null   str           
 7   conf_modelo_a   5000 non-null   float64       
 8   pred_modelo_b   5000 non-null   str           
 9   conf_modelo_b   5000 non-null   float64       
 10  texto_len       5000 non-null   int64         
dtypes: datetime64[us](1), float64(2), int64(2), str(6)
memory usage: 429.8 KB


,id_chamado,data_abertura,bairro,canal,texto,categoria_real,pred_modelo_a,conf_modelo_a,pred_modelo_b,conf_modelo_b,texto_len
0,2026003525,2026-01-16,Botafogo,app_1746,Festas com som automotivo ocorrem toda madruga...,barulho_perturbacao,barulho_perturbacao,0.939,barulho_perturbacao,0.883,147
1,2026001235,2026-06-20,Ilha do Governador,telefone_1746,Festas com som automotivo ocorrem toda madruga...,barulho_perturbacao,barulho_perturbacao,0.927,barulho_perturbacao,0.975,152
2,2026003000,2026-02-17,Campo Grande,portal_web,Solicito reparo urgente na iluminação pública ...,iluminacao_publica,iluminacao_publica,0.947,iluminacao_publica,0.829,173
3,2026002594,2026-01-15,Tijuca,app_1746,A luminária do poste da Rua Dias da Cruz esqui...,iluminacao_publica,coleta_lixo,0.964,iluminacao_publica,0.882,159
4,2026003670,2026-01-07,Penha,app_1746,Solicito remoção de entulho e lixo acumulado n...,coleta_lixo,buraco_via,0.924,coleta_lixo,0.765,148
